# Disease prevalence by region (England vs Scotland/Wales)

Load `ukb_disease_england.csv` and `ukb_disease_scotland_wales.csv` and compute the **distribution** of time-to-diagnosis bins for each disease in each region.

**Bins (proportion of all patients):**
- **0 yr**: prevalent at baseline — `disease == 1` if a binary column exists, else `{disease}_time_to_diagnosis` not NA and ≤ 0
- **0–1 yr**: `disease == 0` and `{disease}_time_to_diagnosis` ≤ 1
- **1–5 yr**: `disease == 0` and 1 < `{disease}_time_to_diagnosis` ≤ 5
- **>5 yr**: `disease == 0` and `{disease}_time_to_diagnosis` > 5

Output: CSV with disease, region (England / Scotland_Wales), and columns for each bin proportion.

In [2]:
import pandas as pd

DIR = "/../../orcd/pool/003/dbertsim_shared/ukb/"

df_england = pd.read_csv(f"{DIR}ukb_disease_england.csv", low_memory=False)
df_scotland_wales = pd.read_csv(f"{DIR}ukb_disease_scotland_wales.csv", low_memory=False)

In [3]:
# Get disease columns: those ending with _time_to_diagnosis
time_cols = [c for c in df_england.columns if c.endswith("_time_to_diagnosis")]
diseases = [c.replace("_time_to_diagnosis", "") for c in time_cols]

def bin_proportions(df, time_col, disease_name):
    """Proportion of all patients in each bin. 0 yr = prevalent (disease==1 or ttd<=0)."""
    ttd = df[time_col]
    # 0 yr: disease == 1 if binary column exists, else ttd notna and ttd <= 0
    if disease_name in df.columns:
        p_0yr = (df[disease_name] == 1).mean()
        mask0 = (df[disease_name] == 0)
        p_0_1yr = (mask0 & ttd.notna() & (ttd <= 1)).mean()
        p_1_5yr = (mask0 & ttd.notna() & (ttd > 1) & (ttd <= 5)).mean()
        p_5yr = (mask0 & ttd.notna() & (ttd > 5)).mean()
    else:
        p_0yr = (ttd.notna() & (ttd <= 0)).mean()
        p_0_1yr = (ttd.notna() & (ttd > 0) & (ttd <= 1)).mean()
        p_1_5yr = (ttd.notna() & (ttd > 1) & (ttd <= 5)).mean()
        p_5yr = (ttd.notna() & (ttd > 5)).mean()
    return p_0yr, p_0_1yr, p_1_5yr, p_5yr

rows = []
for disease, tcol in zip(diseases, time_cols):
    for region_name, df in [("England", df_england), ("Scotland_Wales", df_scotland_wales)]:
        p0, p01, p15, p5 = bin_proportions(df, tcol, disease)
        rows.append({
            "disease": disease,
            "region": region_name,
            "p_0yr": p0,
            "p_0_1yr": p01,
            "p_1_5yr": p15,
            "p_5yr": p5,
        })
result = pd.DataFrame(rows)
result

,disease,england_prevalence,scotland_wales_prevalence
0,acute_kidney_injury,0.064931,0.055337
1,alzheimers_disease,0.014484,0.015884
2,atrial_fibrillation,0.095439,0.093083
3,chronic_kidney_disease,0.082025,0.037746
4,copd,0.065337,0.059095
5,end_stage_renal_disease,0.010997,0.012980
6,heart_failure,0.054576,0.052263
7,hypertensive_heart_kidney_diseases,0.011938,0.007857
8,ischemic_heart_disease,0.146827,0.148762
9,liver_disease,0.047431,0.037404


In [4]:
out_path = f"{DIR}disease_prevalence_by_region.csv"
result.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to disease_prevalence_by_region.csv
